# Sentiment-Enhanced Volatility Prediction

A quantitative finance project combining **LLM-powered sentiment analysis** with **statistical and deep learning volatility models** to generate trading signals.

## Pipeline Overview
```
Financial News -> LLM Sentiment Analysis
                         |
Market Data -> Realized Volatility -> GARCH + LSTM Models
                         |
Sentiment + Volatility -> Trading Strategy -> Backtest
```

### 1. Setup & Imports

In [ ]:
import sys
from pathlib import Path

# Add project root to path
sys.path.insert(0, str(Path.cwd().parent))

import warnings
warnings.filterwarnings('ignore')
import logging
logging.disable(logging.CRITICAL)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('darkgrid')
%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 5)
print('Setup complete')

### 2. Market Data & Realized Volatility

In [ ]:
from data.scrapers.market_data import MarketDataFetcher
from volatility.realized_vol import RealizedVolatility

# Fetch SPY data
m = MarketDataFetcher(tickers=['SPY'])
prices_dict = m.download_prices()
spy = prices_dict['SPY']

print(f'Date range: {spy.index[0].date()} to {spy.index[-1].date()}')
print(f'Trading days: {len(spy)}')
print(f'Current price: ${spy["close"].iloc[-1]:.2f}')
spy.tail()

In [ ]:
# Plot price with volatility regimes
rv = RealizedVolatility.compute_all(spy)

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

axes[0].plot(spy.index, spy['close'], 'b-', lw=1.5)
axes[0].set_title('SPY Price', fontsize=13)
axes[0].set_ylabel('Price ($)')
axes[0].grid(True, alpha=0.3)

for col in rv.columns:
    axes[1].plot(rv.index, rv[col], lw=1.2, alpha=0.8, label=col.replace('_',' ').title())
axes[1].set_title('Realized Volatility Estimators', fontsize=13)
axes[1].set_ylabel('Ann. Volatility')
axes[1].legend(loc='upper right')
axes[1].grid(True, alpha=0.3)

returns = np.log(spy['close'] / spy['close'].shift(1))
axes[2].bar(returns.index, returns, color='steelblue', alpha=0.5, width=1)
axes[2].set_title('Daily Log Returns', fontsize=13)
axes[2].set_ylabel('Return')
axes[2].set_xlabel('Date')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 3. Sentiment Analysis

Comparing **LLM-powered sentiment** (with rule-based fallback) against **VADER lexicon-based** baseline.

In [ ]:
from data.scrapers.news_scraper import NewsScraper
from sentiment.llm_sentiment import LLMSentimentAnalyzer
from sentiment.traditional_sentiment import TraditionalSentimentAnalyzer

# Get sample headlines
headlines = NewsScraper.sample_headlines_for_llm(30)
print(f'Analyzing {len(headlines)} headlines...')

# LLM sentiment (using rule-based fallback)
llm = LLMSentimentAnalyzer()
llm_df = llm.analyze_headlines(headlines, use_api=False)

# Traditional sentiment (VADER)
trad = TraditionalSentimentAnalyzer()
trad_df = trad.analyze_headlines(headlines)

print(f'LLM avg score: {llm_df["llm_score"].mean():.3f}')
print(f'VADER avg score: {trad_df["vader_compound"].mean():.3f}')
print(f'Bullish: {(llm_df["llm_direction"]=="bullish").mean():.1%}')
print(f'Bearish: {(llm_df["llm_direction"]=="bearish").mean():.1%}')

# Combine for comparison
comparison = pd.DataFrame({
    'headline': llm_df['headline'],
    'llm_score': llm_df['llm_score'],
    'vader_score': trad_df['vader_compound'],
    'direction': llm_df['llm_direction'],
    'topic': llm_df['llm_topic'],
})
comparison.head(10)

In [ ]:
# Sentiment distribution comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(llm_df['llm_score'], bins=20, alpha=0.7, color='royalblue', label='LLM')
axes[0].hist(trad_df['vader_compound'], bins=20, alpha=0.5, color='lightcoral', label='VADER')
axes[0].set_title('Score Distribution')
axes[0].set_xlabel('Sentiment Score')
axes[0].legend()

# Direction breakdown
llm_df['llm_direction'].value_counts().plot(kind='bar', ax=axes[1], color=['#2ecc71', '#95a5a6', '#e74c3c'])
axes[1].set_title('LLM Direction Breakdown')
axes[1].set_ylabel('Count')

# Topic distribution
llm_df['llm_topic'].value_counts().plot(kind='bar', ax=axes[2], color=sns.color_palette('viridis', 8))
axes[2].set_title('Topic Distribution')
axes[2].set_ylabel('Count')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

### 4. GARCH Volatility Modeling

In [ ]:
from volatility.garch import GARCHModel

# Fit GARCH(1,1)
ret = np.log(spy['close'] / spy['close'].shift(1)).dropna()
garch11 = GARCHModel(p=1, q=1, model_type='GARCH', distribution='normal')
garch_result = garch11.fit(ret)

print('GARCH(1,1) Results:')
print(f'  AIC: {garch_result["aic"]:.2f}')
print(f'  BIC: {garch_result["bic"]:.2f}')
print(f'  Parameters:')
for k, v in garch_result['params'].items():
    print(f'    {k}: {v:.6f}')

# Model comparison
comparison = GARCHModel.compare_models(ret)
comparison

In [ ]:
# Plot GARCH conditional volatility vs realized
fig, ax = plt.subplots(figsize=(14, 5))

cond_vol = garch_result['conditional_volatility']
yz_vol = rv['yang_zhang'].loc[cond_vol.index]

ax.plot(cond_vol.index, cond_vol, 'b-', lw=1.5, label='GARCH Conditional Vol')
ax.plot(yz_vol.index, yz_vol, 'orange', lw=1.2, alpha=0.7, linestyle='--', label='Realized Vol (YZ)')
ax.set_title('GARCH(1,1) Conditional Volatility vs Realized Volatility', fontsize=13)
ax.set_ylabel('Annualized Volatility')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

### 5. Strategy Backtest

Comparing volatility-based trading strategies.

In [ ]:
from strategy.vol_strategy import VolatilityStrategy
import numpy as np

# Prepare data
vol = rv['yang_zhang'].dropna()
prices = spy['close'].loc[vol.index]

# Create synthetic sentiment aligned with price dates
np.random.seed(42)
dummy_sentiment = pd.Series(np.random.randn(len(vol)) * 0.3, index=vol.index)

# Compare strategies
comparison = VolatilityStrategy.compare_strategies(vol, prices, dummy_sentiment)
comparison

In [ ]:
# Run combined strategy backtest
combined = VolatilityStrategy(strategy_type='combined')
result = combined.backtest(vol, prices, sentiment=dummy_sentiment)
metrics = result['metrics']

print('Combined Strategy Performance:')
print(f'  Total Return: {metrics.total_return_pct:.2f}%')
print(f'  Sharpe Ratio: {metrics.sharpe_ratio:.2f}')
print(f'  Max Drawdown: {metrics.max_drawdown:.2%}')
print(f'  Win Rate: {metrics.win_rate:.1%}')
print(f'  Total Trades: {metrics.total_trades}')
print(f'  Profit Factor: {metrics.profit_factor:.2f}')

# Display equity curve
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(metrics.equity_curve.index, metrics.equity_curve, 'b-', lw=1.5)
axes[0].axhline(y=1_000_000, color='gray', linestyle='--', alpha=0.5, label='Initial Capital')
axes[0].set_title('Equity Curve', fontsize=13)
axes[0].set_ylabel('Portfolio Value ($)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

cummax = metrics.equity_curve.cummax()
drawdown = (metrics.equity_curve - cummax) / cummax
axes[1].fill_between(drawdown.index, 0, drawdown * 100, color='red', alpha=0.3)
axes[1].set_title('Drawdown (%)', fontsize=13)
axes[1].set_ylabel('Drawdown %')
axes[1].set_xlabel('Date')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 6. End-to-End Pipeline

In [ ]:
from app.pipeline import Pipeline

# Run full pipeline
p = Pipeline(ticker='SPY')
results = p.run_all()

print('=== Pipeline Results ===')
print(f"Data: {results['data']['n_observations']} trading days")
print(f"Sentiment: {results['sentiment']['n_headlines']} headlines analyzed")
print(f"GARCH AIC: {results['volatility']['garch_aic']:.2f}")
print(f"Best Strategy: {results['strategy']['best_strategy']} ")
print(f"  Sharpe Ratio: {results['strategy']['best_sharpe']:.2f}")

### 7. Run the Dashboard

```bash
cd sentiment-vol-lab
streamlit run app/app.py
```

---

### Summary

This notebook demonstrated the full pipeline:
1. **Market Data**: Downloaded SPY data and computed multiple realized volatility estimators
2. **Sentiment Analysis**: Compared LLM-powered vs VADER sentiment on financial headlines
3. **Volatility Models**: Fit GARCH(1,1) and compared across model specifications
4. **Strategy Backtest**: Tested volatility-based trading strategies with performance metrics

The project architecture is modular and extensible:
- Add real LLM sentiment by setting `OPENAI_API_KEY`
- Connect to real-time data sources via yfinance or WebSocket feeds
- Extend with additional volatility models (HAR-RV, Neural GARCH)
- Add more sophisticated trading strategies